# German Credit 评分卡演示\n
\n
本 Notebook 演示完整流程：\n
1. 加载 germancredit 数据\n
2. 分箱与 WOE 转换\n
3. 训练 ScoreCard（descending/ascending）\n
4. 校验分数方向与概率关系\n
5. 查看 score_odds_reference 基准分参照\n

In [1]:
import os, sys
sys.path.append('../')

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from hscredit.core.binning import OptimalBinning
from hscredit.core.models import ScoreCard
from hscredit.utils.datasets import germancredit

In [2]:
def summarize_direction_result(name: str, y_true: pd.Series, scores: np.ndarray, proba: np.ndarray) -> None:
    good_mean = float(scores[y_true == 0].mean())
    bad_mean = float(scores[y_true == 1].mean())
    corr = float(np.corrcoef(scores, proba)[0, 1])

    print(f'[{name}]')
    print(f'score range       : [{scores.min():.2f}, {scores.max():.2f}]')
    print(f'mean score good=0 : {good_mean:.2f}')
    print(f'mean score bad=1  : {bad_mean:.2f}')
    print(f'corr(score, pbad) : {corr:.4f}')

    if name == 'descending':
        print(f'expect good > bad : {good_mean > bad_mean}')
        print(f'expect corr < 0   : {corr < 0}')
    else:
        print(f'expect bad > good : {bad_mean > good_mean}')
        print(f'expect corr > 0   : {corr > 0}')

## 1) 加载数据

In [3]:
df = germancredit().copy()
target_col = 'class'
y = df[target_col].astype(int)
X = df.drop(columns=[target_col])

print('German Credit loaded')
print(f'shape             : {df.shape}')
print(f'features          : {X.shape[1]}')
print(f'bad rate (class=1): {y.mean():.4f}')
df.head()

German Credit loaded
shape             : (1000, 21)
features          : 20
bad rate (class=1): 0.3000


,status_of_existing_checking_account,duration_in_month,credit_history,purpose,credit_amount,savings_account_and_bonds,present_employment_since,installment_rate_in_percentage_of_disposable_income,personal_status_and_sex,other_debtors_or_guarantors,...,property,age_in_years,other_installment_plans,housing,number_of_existing_credits_at_this_bank,job,number_of_people_being_liable_to_provide_maintenance_for,telephone,foreign_worker,class
0,... < 0 DM,6,critical account/ other credits existing (not ...,radio/television,1169,unknown/ no savings account,... >= 7 years,4,male : divorced/separated,none,...,real estate,67,none,own,2,skilled employee / official,1,"yes, registered under the customers name",yes,0
1,0 <= ... < 200 DM,48,existing credits paid back duly till now,radio/television,5951,... < 100 DM,1 <= ... < 4 years,2,male : divorced/separated,none,...,real estate,22,none,own,1,skilled employee / official,1,none,yes,1
2,no checking account,12,critical account/ other credits existing (not ...,education,2096,... < 100 DM,4 <= ... < 7 years,2,male : divorced/separated,none,...,real estate,49,none,own,1,unskilled - resident,2,none,yes,0
3,... < 0 DM,42,existing credits paid back duly till now,furniture/equipment,7882,... < 100 DM,4 <= ... < 7 years,2,male : divorced/separated,guarantor,...,building society savings agreement/ life insur...,45,none,for free,1,skilled employee / official,2,none,yes,0
4,... < 0 DM,24,delay in paying off in the past,car (new),4870,... < 100 DM,1 <= ... < 4 years,3,male : divorced/separated,none,...,unknown / no property,53,none,for free,2,skilled employee / official,2,none,yes,1


## 2) 划分训练测试 + 分箱与 WOE

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y,
)

binner = OptimalBinning(method='target_bad_rate', max_n_bins=5)
binner.fit(X_train, y_train)

X_train_woe = binner.transform(X_train, metric='woe')
X_test_woe = binner.transform(X_test, metric='woe')

print('X_train_woe shape:', X_train_woe.shape)
X_train_woe.head()

X_train_woe shape: (700, 20)


,status_of_existing_checking_account,duration_in_month,credit_history,purpose,credit_amount,savings_account_and_bonds,present_employment_since,installment_rate_in_percentage_of_disposable_income,personal_status_and_sex,other_debtors_or_guarantors,present_residence_since,property,age_in_years,other_installment_plans,housing,number_of_existing_credits_at_this_bank,job,number_of_people_being_liable_to_provide_maintenance_for,telephone,foreign_worker
10,0.415165,-0.344096,0.082842,0.173721,-0.043939,0.301787,0.579034,0.100083,-0.287682,-0.005997,0.071459,0.034804,0.455256,-0.174622,0.468861,-0.004193,-0.039556,-0.005666,0.044150,0.044951
82,-1.184134,0.101228,0.082842,0.173721,-0.043939,0.098061,-0.074691,0.100083,-0.114603,-0.005997,-0.010989,-0.051449,0.455256,-0.174622,0.468861,-0.004193,-0.104261,-0.005666,0.044150,0.044951
827,-1.184134,0.101228,1.134980,0.173721,-0.043939,0.301787,-0.074691,-0.211784,0.069593,-0.005997,-0.010989,0.034804,-0.264053,0.308301,-0.214468,-0.004193,-0.039556,0.029853,0.044150,0.044951
410,0.415165,0.101228,0.082842,-0.384846,-0.043939,0.301787,-0.194156,0.149489,0.069593,-0.005997,-0.010989,0.034804,0.455256,-0.174622,-0.214468,-0.004193,-0.039556,-0.005666,-0.066521,0.044951
48,-1.184134,-0.344096,-0.699082,0.173721,-0.043939,0.301787,-0.074691,-0.292136,-0.287682,-0.005997,-0.010989,-0.051449,-0.264053,-0.174622,-0.214468,-0.004193,-0.104261,-0.005666,0.044150,0.044951


## 3) 信用分模式（descending）

In [5]:
scorecard_desc = ScoreCard(
    pdo=60,
    rate=2,
    base_odds=35,
    base_score=750,
    direction='descending',
    binner=binner,
)
scorecard_desc.fit(X_train_woe, y_train, input_type='woe')

proba_desc = scorecard_desc.predict_proba(X_test)[:, 1]
score_desc = scorecard_desc.predict(X_test, input_type='raw')

auc_desc = roc_auc_score(y_test, proba_desc)
print(f'AUC (descending): {auc_desc:.4f}')
summarize_direction_result('descending', y_test, score_desc, proba_desc)

AUC (descending): 0.8028
[descending]
score range       : [238.34, 2287.31]
mean score good=0 : 609.38
mean score bad=1  : 452.91
corr(score, pbad) : -0.6518
expect good > bad : True
expect corr < 0   : True


## 4) 欺诈分模式（ascending）

In [6]:
scorecard_asc = ScoreCard(
    pdo=60,
    rate=2,
    base_odds=35,
    base_score=750,
    direction='ascending',
    binner=binner,
)
scorecard_asc.fit(X_train_woe, y_train, input_type='woe')

proba_asc = scorecard_asc.predict_proba(X_test)[:, 1]
score_asc = scorecard_asc.predict(X_test, input_type='raw')

auc_asc = roc_auc_score(y_test, proba_asc)
print(f'AUC (ascending): {auc_asc:.4f}')
summarize_direction_result('ascending', y_test, score_asc, proba_asc)

AUC (ascending): 0.8028
[ascending]
score range       : [-787.31, 1261.66]
mean score good=0 : 890.62
mean score bad=1  : 1047.09
corr(score, pbad) : 0.6518
expect bad > good : True
expect corr > 0   : True


In [7]:
scorecard_asc.scorecard_points()

,变量名称,变量含义,变量分箱,对应分数,WOE值
0,基础分,截距项（基准分数）,-,515.41,NaN
1,status_of_existing_checking_account,,no checking account,79.99,-1.1841
2,status_of_existing_checking_account,,... >= 200 DM / salary assignments for at leas...,9.02,-0.1335
3,status_of_existing_checking_account,,0 <= ... < 200 DM,-28.04,0.4152
4,status_of_existing_checking_account,,... < 0 DM,-51.60,0.7639
...,...,...,...,...,...
69,number_of_people_being_liable_to_provide_maint...,,"[1.5, +inf)",-0.18,0.0299
70,telephone,,"yes, registered under the customers name",2.64,-0.0665
71,telephone,,none,-1.75,0.0442
72,foreign_worker,,no,122.86,-1.7177


## 5) WOE/LR 系数检查

In [8]:
coef = scorecard_desc.coef_
coef_series = pd.Series(coef, index=X_train_woe.columns)
negative_coef_features = coef_series[coef_series < 0].sort_values()
coef_positive_ratio = float((coef > 0).sum() / len(coef))

print('[WOE/LR sanity]')
print(f'coef count        : {len(coef)}')
print(f'coef > 0 ratio    : {coef_positive_ratio:.4f}')
print(f'all coef > 0      : {bool(np.all(coef > 0))}')
if len(negative_coef_features) > 0:
    print('negative coef feats:')
    print(negative_coef_features.to_string())

[WOE/LR sanity]
coef count        : 20
coef > 0 ratio    : 1.0000
all coef > 0      : True


## 6) score_odds_reference 基准分检查

In [9]:
ref = scorecard_desc.score_odds_reference
ref_row = ref.iloc[(ref['评分'] - 750).abs().argsort()[:1]]
ref_row[['评分', '理论逾期率(%)', '好客户:坏客户']]

,评分,理论逾期率(%),好客户:坏客户
50,750,2.7778%,35.0:1


In [10]:
scorecard_desc.score_odds_reference

,评分,理论Odds(坏好比),好客户:坏客户,理论逾期率,理论逾期率(%),对数Odds
0,1050,0.0009,1120.0:1,0.000892,0.0892%,-7.0211
1,1044,0.0010,1045.0:1,0.000956,0.0956%,-6.9518
2,1038,0.0010,975.0:1,0.001025,0.1025%,-6.8825
3,1032,0.0011,909.7:1,0.001098,0.1098%,-6.8131
4,1026,0.0012,848.8:1,0.001177,0.1177%,-6.7438
...,...,...,...,...,...,...
96,474,0.6929,1.4:1,0.409297,40.9297%,-0.3669
97,468,0.7426,1.3:1,0.426155,42.6155%,-0.2976
98,462,0.7959,1.3:1,0.443186,44.3186%,-0.2282
99,456,0.8531,1.2:1,0.460352,46.0352%,-0.1589


In [11]:
scorecard_asc.score_odds_reference

,评分,理论Odds(坏好比),好客户:坏客户,理论逾期率,理论逾期率(%),对数Odds
0,450,0.0009,1120.0:1,0.000892,0.0892%,-7.0211
1,456,0.0010,1045.0:1,0.000956,0.0956%,-6.9518
2,462,0.0010,975.0:1,0.001025,0.1025%,-6.8825
3,468,0.0011,909.7:1,0.001098,0.1098%,-6.8131
4,474,0.0012,848.8:1,0.001177,0.1177%,-6.7438
...,...,...,...,...,...,...
96,1026,0.6929,1.4:1,0.409297,40.9297%,-0.3669
97,1032,0.7426,1.3:1,0.426155,42.6155%,-0.2976
98,1038,0.7959,1.3:1,0.443186,44.3186%,-0.2282
99,1044,0.8531,1.2:1,0.460352,46.0352%,-0.1589


## 7) 使用 feature_bin_stats 查看信用分与欺诈分效果

分别对 `descending` 信用评分和 `ascending` 欺诈评分做分箱效果统计。

In [12]:
from hscredit.report import feature_bin_stats

score_effect_df = pd.DataFrame({
    'target': y_test.to_numpy(),
    '信用评分': score_desc,
    '欺诈评分': score_asc,
}, index=X_test.index)

score_effect_df.head()

,target,信用评分,欺诈评分
115,0,741.039778,758.960222
346,0,543.981948,956.018052
328,0,512.789109,987.210891
974,0,609.768628,890.231372
587,0,457.494693,1042.505307


In [13]:
credit_score_stats = feature_bin_stats(
    score_effect_df,
    '信用评分',
    target='target',
    method='quantile',
    max_n_bins=10,
    min_bin_size=0.05,
    margins=True,
    return_cols=['样本总数', '样本占比', '坏样本数', '坏样本率', 'LIFT值', '累积LIFT值', '分档KS值'],
)

display(credit_score_stats)

,指标名称,指标含义,分箱标签,样本总数,样本占比,坏样本数,坏样本率,LIFT值,累积LIFT值,分档KS值
0,信用评分,信用评分,"[-inf, 380.1922)",30,0.10,22,0.733333,2.4444,2.4444,0.206349
1,信用评分,信用评分,"[380.1922, 473.502)",60,0.20,32,0.533333,1.7778,2.0000,0.428571
2,信用评分,信用评分,"[473.502, 504.8349)",30,0.10,11,0.366667,1.2222,1.8056,0.460317
3,信用评分,信用评分,"[504.8349, 521.6464)",15,0.05,8,0.533333,1.7778,1.8025,0.515873
4,信用评分,信用评分,"[521.6464, 543.9836)",15,0.05,3,0.200000,0.6667,1.6889,0.492063
5,信用评分,信用评分,"[543.9836, 569.4105)",30,0.10,1,0.033333,0.1111,1.4259,0.365079
6,信用评分,信用评分,"[569.4105, 600.1012)",15,0.05,6,0.400000,1.3333,1.4188,0.388889
7,信用评分,信用评分,"[600.1012, 649.5297)",45,0.15,2,0.044444,0.1481,1.1806,0.206349
8,信用评分,信用评分,"[649.5297, 681.5564)",15,0.05,3,0.200000,0.6667,1.1503,0.182540
9,信用评分,信用评分,"[681.5564, +inf)",45,0.15,2,0.044444,0.1481,1.0000,0.000000


In [14]:
fraud_score_stats = feature_bin_stats(
    score_effect_df,
    '欺诈评分',
    target='target',
    method='quantile',
    max_n_bins=10,
    min_bin_size=0.05,
    margins=True,
    return_cols=['样本总数', '样本占比', '坏样本数', '坏样本率', 'LIFT值', '累积LIFT值', '分档KS值'],
)

display(fraud_score_stats)

,指标名称,指标含义,分箱标签,样本总数,样本占比,坏样本数,坏样本率,LIFT值,累积LIFT值,分档KS值
0,欺诈评分,欺诈评分,"[-inf, 818.4436)",45,0.15,2,0.044444,0.1481,0.1481,0.182540
1,欺诈评分,欺诈评分,"[818.4436, 850.4703)",15,0.05,3,0.200000,0.6667,0.2778,0.206349
2,欺诈评分,欺诈评分,"[850.4703, 899.8988)",45,0.15,2,0.044444,0.1481,0.2222,0.388889
3,欺诈评分,欺诈评分,"[899.8988, 930.5895)",15,0.05,6,0.400000,1.3333,0.3611,0.365079
4,欺诈评分,欺诈评分,"[930.5895, 956.0164)",30,0.10,1,0.033333,0.1111,0.3111,0.492063
5,欺诈评分,欺诈评分,"[956.0164, 978.3536)",15,0.05,3,0.200000,0.6667,0.3434,0.515873
6,欺诈评分,欺诈评分,"[978.3536, 995.1651)",15,0.05,8,0.533333,1.7778,0.4630,0.460317
7,欺诈评分,欺诈评分,"[995.1651, 1026.498)",30,0.10,11,0.366667,1.2222,0.5714,0.428571
8,欺诈评分,欺诈评分,"[1026.498, 1119.8078)",60,0.20,32,0.533333,1.7778,0.8395,0.206349
9,欺诈评分,欺诈评分,"[1119.8078, +inf)",30,0.10,22,0.733333,2.4444,1.0000,0.000000


In [15]:
from pathlib import Path
import tempfile
import traceback

scorecard_export_root = Path(tempfile.mkdtemp(prefix='hscredit_scorecard_exports_'))
scorecard_export_root

WindowsPath('C:/Users/18306/AppData/Local/Temp/hscredit_scorecard_exports_ib_sq9v6')

## 8) scorecard 导出与离线一致性验证

依次验证以下导出路径：

1. `pickle` 离线模型文件
2. `json` 规则文件
3. Python 部署代码
4. PMML 文件

说明：
- `pickle` 期望与原模型完全一致。
- `json` 规则导出当前按规则分数字典回载，允许存在很小的精度损失。
- Python 部署代码使用导出的规则直接对原始样本打分，理论上应与原模型几乎一致。
- PMML 依赖 `java`、`sklearn-pandas`、`sklearn2pmml`、`pypmml`；若环境仍缺少任一依赖，会在结果里记录原因。

In [16]:
# 为避免在 Windows/GBK 环境下 sklearn2pmml/Java 子进程输出导致的 UTF-8 解码错误，
# 在导出 PMML 前强制 Java 使用 UTF-8，并开启 Python UTF-8 模式。
import os

flags = ['-Dfile.encoding=UTF-8', '-Dsun.jnu.encoding=UTF-8']
opts = os.environ.get('JAVA_TOOL_OPTIONS', '')
for f in flags:
    if f not in opts:
        opts = (opts + ' ' + f).strip()
os.environ['JAVA_TOOL_OPTIONS'] = opts
os.environ['PYTHONUTF8'] = '1'

print('JAVA_TOOL_OPTIONS =', os.environ['JAVA_TOOL_OPTIONS'])
print('PYTHONUTF8 =', os.environ['PYTHONUTF8'])

JAVA_TOOL_OPTIONS = -Dfile.encoding=UTF-8 -Dsun.jnu.encoding=UTF-8
PYTHONUTF8 = 1


In [17]:
from pathlib import Path
import tempfile
import traceback

export_summary_rows = []
export_artifacts = {}
scorecard_export_root = Path(tempfile.mkdtemp(prefix='hscredit_scorecard_exports_'))

scorecard_models = {
    'descending': scorecard_desc,
}

if 'scorecard_asc' in globals():
    scorecard_models['ascending'] = scorecard_asc
else:
    scorecard_models['ascending'] = ScoreCard(
        pdo=60,
        rate=2,
        base_odds=35,
        base_score=750,
        direction='ascending',
        binner=binner,
    )
    scorecard_models['ascending'].fit(X_train_woe, y_train, input_type='woe')

for model_name, card in scorecard_models.items():
    model_dir = scorecard_export_root / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    reference_scores = card.predict(X_test, input_type='raw')
    model_results = {}

    pickle_path = model_dir / 'scorecard.pkl'
    card.save_pickle(str(pickle_path))
    pickle_card = ScoreCard.load_pickle(str(pickle_path))
    pickle_scores = pickle_card.predict(X_test, input_type='raw')
    model_results['pickle'] = {
        'path': str(pickle_path),
        'max_abs_diff': float(np.max(np.abs(reference_scores - pickle_scores))),
        'mean_abs_diff': float(np.mean(np.abs(reference_scores - pickle_scores))),
        'status': 'ok',
    }

    json_path = model_dir / 'scorecard.json'
    card.export(to_json=str(json_path), include_meta=True)
    json_card = ScoreCard(binner=binner)
    json_card.load(str(json_path))
    json_scores = json_card.predict(X_test, input_type='raw')
    model_results['json'] = {
        'path': str(json_path),
        'max_abs_diff': float(np.max(np.abs(reference_scores - json_scores))),
        'mean_abs_diff': float(np.mean(np.abs(reference_scores - json_scores))),
        'status': 'ok',
    }

    py_path = model_dir / 'scorecard_deploy.py'
    py_namespace = {}
    py_code = card.export_deployment_code(language='python', output_file=str(py_path), decimal=12)
    exec(py_code, py_namespace)
    py_scores = X_test.apply(lambda row: py_namespace['calculate_score'](row.to_dict()), axis=1).to_numpy()
    model_results['python'] = {
        'path': str(py_path),
        'max_abs_diff': float(np.max(np.abs(reference_scores - py_scores))),
        'mean_abs_diff': float(np.mean(np.abs(reference_scores - py_scores))),
        'status': 'ok',
    }

    pmml_path = model_dir / 'scorecard.pmml'
    try:
        from pypmml import Model

        card.export_pmml(str(pmml_path))
        pmml_model = Model.load(str(pmml_path))
        pmml_output = pmml_model.predict(X_test)
        pmml_score_col = 'score' if 'score' in pmml_output.columns else pmml_output.columns[-1]
        pmml_scores = pmml_output[pmml_score_col].to_numpy(dtype=float)
        model_results['pmml'] = {
            'path': str(pmml_path),
            'score_column': pmml_score_col,
            'max_abs_diff': float(np.max(np.abs(reference_scores - pmml_scores))),
            'mean_abs_diff': float(np.mean(np.abs(reference_scores - pmml_scores))),
            'status': 'ok',
        }
    except Exception as exc:
        model_results['pmml'] = {
            'path': str(pmml_path),
            'status': 'failed',
            'error_type': type(exc).__name__,
            'error': str(exc),
            'traceback_tail': traceback.format_exc().splitlines()[-5:],
        }

    export_artifacts[model_name] = model_results

    for export_type, result in model_results.items():
        export_summary_rows.append({
            'model': model_name,
            'export_type': export_type,
            'status': result['status'],
            'max_abs_diff': result.get('max_abs_diff'),
            'mean_abs_diff': result.get('mean_abs_diff'),
            'path': result['path'],
            'note': result.get('error', ''),
        })

export_summary_df = pd.DataFrame(export_summary_rows)
export_summary_df

模型已保存至: C:\Users\18306\AppData\Local\Temp\hscredit_scorecard_exports_wrzqjv_q\descending\scorecard.pkl


Exception in thread Thread-8:
Traceback (most recent call last):
  File "C:\Users\18306\anaconda3\lib\threading.py", line 980, in _bootstrap_inner
    self.run()
  File "C:\Users\18306\anaconda3\lib\threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\18306\anaconda3\lib\subprocess.py", line 1479, in _readerthread
    buffer.append(fh.read())
UnicodeDecodeError: 'gbk' codec can't decode byte 0x88 in position 79: illegal multibyte sequence


PMML 文件已导出至: C:\Users\18306\AppData\Local\Temp\hscredit_scorecard_exports_wrzqjv_q\descending\scorecard.pmml
模型已保存至: C:\Users\18306\AppData\Local\Temp\hscredit_scorecard_exports_wrzqjv_q\ascending\scorecard.pkl


Exception in thread Thread-14:
Traceback (most recent call last):
  File "C:\Users\18306\anaconda3\lib\threading.py", line 980, in _bootstrap_inner
    self.run()
  File "C:\Users\18306\anaconda3\lib\threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\18306\anaconda3\lib\subprocess.py", line 1479, in _readerthread
    buffer.append(fh.read())
UnicodeDecodeError: 'gbk' codec can't decode byte 0x88 in position 79: illegal multibyte sequence


PMML 文件已导出至: C:\Users\18306\AppData\Local\Temp\hscredit_scorecard_exports_wrzqjv_q\ascending\scorecard.pmml


,model,export_type,status,max_abs_diff,mean_abs_diff,path,note
0,descending,pickle,ok,0.000000e+00,0.000000e+00,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,
1,descending,json,ok,2.019218e+01,5.420390e+00,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,
2,descending,python,ok,3.524292e-12,1.645617e-12,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,
3,descending,pmml,ok,4.349400e-02,1.602417e-02,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,
4,ascending,pickle,ok,0.000000e+00,0.000000e+00,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,
5,ascending,json,ok,2.019218e+01,5.420390e+00,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,
6,ascending,python,ok,3.524292e-12,1.625343e-12,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,
7,ascending,pmml,ok,4.349400e-02,1.602417e-02,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,


In [18]:
diagnostic_rows = []

for model_name, card in scorecard_models.items():
    py_namespace = {}
    exec(card.export_deployment_code(language='python', decimal=12), py_namespace)
    reference_scores = card.predict(X_test, input_type='raw')
    py_scores = X_test.apply(lambda row: py_namespace['calculate_score'](row.to_dict()), axis=1).to_numpy()
    diffs = np.abs(reference_scores - py_scores)
    max_idx = int(np.argmax(diffs))
    diagnostic_rows.append({
        'model': model_name,
        'row_index': int(X_test.index[max_idx]),
        'max_abs_diff': float(diffs[max_idx]),
        'reference_score': float(reference_scores[max_idx]),
        'python_score': float(py_scores[max_idx]),
    })

pd.DataFrame(diagnostic_rows)

,model,row_index,max_abs_diff,reference_score,python_score
0,descending,794,3.524292e-12,587.797087,587.797087
1,ascending,794,3.524292e-12,912.202913,912.202913


In [19]:
debug_feature_rows = []

for model_name, row_index in [('descending', 856), ('ascending', 431)]:
    card = scorecard_models[model_name]
    row = X_test.loc[[row_index]]
    x_woe = binner.transform(row, metric='woe')[card.feature_names_]
    sub_scores = card._woe_to_score(x_woe, card.feature_names_)[0]

    for feature, ref_sub_score in zip(card.feature_names_, sub_scores):
        rule = card.rules_[feature]
        selected_score = None
        raw_value = row.iloc[0][feature]
        for label, score in zip(rule['bin_labels'], rule['scores']):
            cond = card._bin_label_to_python_condition('val', label)
            if eval(cond, {'np': np}, {'val': raw_value}):
                selected_score = float(score)
                break

        if selected_score is None or abs(selected_score - ref_sub_score) > 1e-9:
            debug_feature_rows.append({
                'model': model_name,
                'row_index': row_index,
                'feature': feature,
                'raw_value': raw_value,
                'reference_sub_score': float(ref_sub_score),
                'selected_score': selected_score,
                'labels': list(rule['bin_labels']),
            })
            break

pd.DataFrame(debug_feature_rows)

,model,row_index,feature,raw_value,reference_sub_score,selected_score,labels
0,descending,856,purpose,retraining,42.162295,None,"[car (used), radio/television, domestic applia..."
1,ascending,431,purpose,others,42.162295,None,"[car (used), radio/television, domestic applia..."


In [20]:
for model_name, results in export_artifacts.items():
    print(f'[{model_name}]')
    for export_type, result in results.items():
        if result['status'] == 'ok':
            print(
                f"  - {export_type:<6} max_abs_diff={result['max_abs_diff']:.12f} "
                f"mean_abs_diff={result['mean_abs_diff']:.12f}"
            )
        else:
            print(f"  - {export_type:<6} failed: {result['error_type']} - {result['error']}")

[descending]
  - pickle max_abs_diff=0.000000000000 mean_abs_diff=0.000000000000
  - json   max_abs_diff=20.192182271686 mean_abs_diff=5.420390360562
  - python max_abs_diff=0.000000000004 mean_abs_diff=0.000000000002
  - pmml   max_abs_diff=0.043493998949 mean_abs_diff=0.016024166726
[ascending]
  - pickle max_abs_diff=0.000000000000 mean_abs_diff=0.000000000000
  - json   max_abs_diff=20.192182271686 mean_abs_diff=5.420390360562
  - python max_abs_diff=0.000000000004 mean_abs_diff=0.000000000002
  - pmml   max_abs_diff=0.043493998949 mean_abs_diff=0.016024166726


In [21]:
export_summary_df.sort_values(['model', 'export_type']).reset_index(drop=True)

,model,export_type,status,max_abs_diff,mean_abs_diff,path,note
0,ascending,json,ok,2.019218e+01,5.420390e+00,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,
1,ascending,pickle,ok,0.000000e+00,0.000000e+00,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,
2,ascending,pmml,ok,4.349400e-02,1.602417e-02,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,
3,ascending,python,ok,3.524292e-12,1.625343e-12,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,
4,descending,json,ok,2.019218e+01,5.420390e+00,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,
5,descending,pickle,ok,0.000000e+00,0.000000e+00,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,
6,descending,pmml,ok,4.349400e-02,1.602417e-02,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,
7,descending,python,ok,3.524292e-12,1.645617e-12,C:\Users\18306\AppData\Local\Temp\hscredit_sco...,


In [22]:
export_conclusion_df = (
    export_summary_df
    .assign(
        acceptable=lambda df: df['max_abs_diff'].fillna(np.inf) <= 0.05,
        interpretation=lambda df: np.where(
            df['status'] != 'ok',
            '导出失败，需要单独排查环境或兼容性问题',
            np.where(
                df['max_abs_diff'].fillna(np.inf) <= 1e-9,
                '与原始评分几乎完全一致',
                np.where(
                    df['max_abs_diff'].fillna(np.inf) <= 0.05,
                    '存在极小精度损失，可接受',
                    '差异偏大，需要继续排查',
                )
            )
        )
    )
    [['model', 'export_type', 'status', 'max_abs_diff', 'mean_abs_diff', 'acceptable', 'interpretation', 'note']]
)

export_conclusion_df.sort_values(['model', 'export_type']).reset_index(drop=True)

,model,export_type,status,max_abs_diff,mean_abs_diff,acceptable,interpretation,note
0,ascending,json,ok,2.019218e+01,5.420390e+00,False,差异偏大，需要继续排查,
1,ascending,pickle,ok,0.000000e+00,0.000000e+00,True,与原始评分几乎完全一致,
2,ascending,pmml,ok,4.349400e-02,1.602417e-02,True,存在极小精度损失，可接受,
3,ascending,python,ok,3.524292e-12,1.625343e-12,True,与原始评分几乎完全一致,
4,descending,json,ok,2.019218e+01,5.420390e+00,False,差异偏大，需要继续排查,
5,descending,pickle,ok,0.000000e+00,0.000000e+00,True,与原始评分几乎完全一致,
6,descending,pmml,ok,4.349400e-02,1.602417e-02,True,存在极小精度损失，可接受,
7,descending,python,ok,3.524292e-12,1.645617e-12,True,与原始评分几乎完全一致,


In [23]:
from pypmml import Model


def _python_scores(card, sample):
    namespace = {}
    exec(card.export_deployment_code(language='python', decimal=12), namespace)
    return sample.apply(lambda row: namespace['calculate_score'](row.to_dict()), axis=1).to_numpy()


def _pmml_scores(sample, model_name):
    try:
        pmml_result = export_artifacts.get(model_name, {}).get('pmml', None)
        if not pmml_result or pmml_result.get('status') != 'ok':
            return None
        pmml_path = pmml_result['path']
        pmml_model = Model.load(pmml_path)
        pmml_output = pmml_model.predict(sample)
        pmml_score_col = 'score' if 'score' in pmml_output.columns else pmml_output.columns[-1]
        return pmml_output[pmml_score_col].to_numpy(dtype=float)
    except Exception:
        return None


def _feature_score_breakdown(card, row_index):
    row_df = X_test.loc[[row_index]]
    x_woe = binner.transform(row_df, metric='woe')[card.feature_names_]
    sub_scores = card._woe_to_score(x_woe, card.feature_names_)[0]
    return pd.DataFrame({
        'feature': card.feature_names_,
        'raw_value': [row_df.iloc[0][feature] for feature in card.feature_names_],
        'woe_score': sub_scores,
        'bin_labels': [list(card.rules_[feature]['bin_labels']) for feature in card.feature_names_],
        'scores': [list(card.rules_[feature]['scores']) for feature in card.feature_names_],
    }).sort_values('woe_score')


# PMML diagnostics (optional, only if PMML export succeeded)
pmml_diag_rows = []
for model_name, card in scorecard_models.items():
    reference_scores = card.predict(X_test, input_type='raw')
    pmml_scores = _pmml_scores(X_test.copy(), model_name)
    if pmml_scores is None:
        continue
    diffs = np.abs(reference_scores - pmml_scores)
    order = np.argsort(diffs)[::-1][:10]
    for pos in order:
        pmml_diag_rows.append({
            'model': model_name,
            'row_index': int(X_test.index[pos]),
            'reference_score': float(reference_scores[pos]),
            'pmml_score': float(pmml_scores[pos]),
            'abs_diff': float(diffs[pos]),
        })
pmml_diag_df = pd.DataFrame(pmml_diag_rows)
if not pmml_diag_df.empty:
    print('PMML top diffs')
    display(pmml_diag_df)

# Python diagnostics
python_diag_rows = []
for model_name, card in scorecard_models.items():
    reference_scores = card.predict(X_test, input_type='raw')
    py_scores = _python_scores(card, X_test.copy())
    diffs = np.abs(reference_scores - py_scores)
    order = np.argsort(diffs)[::-1][:10]
    for pos in order:
        python_diag_rows.append({
            'model': model_name,
            'row_index': int(X_test.index[pos]),
            'reference_score': float(reference_scores[pos]),
            'python_score': float(py_scores[pos]),
            'abs_diff': float(diffs[pos]),
        })
python_diag_df = pd.DataFrame(python_diag_rows)

print('Python top diffs')
display(python_diag_df)

# Build focus rows from available diagnostics
_sources = []
if 'pmml_diag_df' in globals() and not pmml_diag_df.empty:
    _sources.append(pmml_diag_df['row_index'].head(5).tolist())
if not python_diag_df.empty:
    _sources.append(python_diag_df['row_index'].head(5).tolist())
focus_rows = sorted(set(sum(_sources, []))) if _sources else []

for row_index in focus_rows[:5]:
    print(f'===== row_index={row_index} descending =====')
    display(_feature_score_breakdown(scorecard_desc, row_index).head(10))
    print(X_test.loc[row_index].to_frame().T)

PMML top diffs


,model,row_index,reference_score,pmml_score,abs_diff
0,descending,683,565.300369,565.256875,0.043494
1,descending,205,429.826846,429.786875,0.039972
2,descending,796,602.685609,602.646875,0.038734
3,descending,395,301.054738,301.016875,0.037864
4,descending,788,319.923930,319.886875,0.037055
5,descending,386,601.713491,601.676875,0.036616
6,descending,800,501.453440,501.416875,0.036566
7,descending,519,793.223370,793.186875,0.036495
8,descending,933,769.712634,769.676875,0.035759
9,descending,896,401.802422,401.766875,0.035548


Python top diffs


,model,row_index,reference_score,python_score,abs_diff
0,descending,794,587.797087,587.797087,3.524292e-12
1,descending,854,495.729850,495.729850,3.410605e-12
2,descending,134,583.046797,583.046797,3.296918e-12
3,descending,563,424.865760,424.865760,3.296918e-12
4,descending,435,643.603257,643.603257,3.296918e-12
5,descending,681,680.282011,680.282011,3.183231e-12
6,descending,921,562.161869,562.161869,3.183231e-12
7,descending,395,301.054738,301.054738,3.126388e-12
8,descending,858,429.326487,429.326487,3.126388e-12
9,descending,168,586.942783,586.942783,3.069545e-12


===== row_index=134 descending =====


...

    status_of_existing_checking_account duration_in_month  \
395                   0 <= ... < 200 DM                39   

                      credit_history    purpose credit_amount  \
395  delay in paying off in the past  education         11760   

    savings_account_and_bonds present_employment_since  \
395       100 <= ... < 500 DM       4 <= ... < 7 years   

    installment_rate_in_percentage_of_disposable_income  \
395                                                  2    

    personal_status_and_sex other_debtors_or_guarantors  \
395           male : single                        none   

    present_residence_since               property age_in_years  \
395                       3  unknown / no property           32   

    other_installment_plans housing number_of_existing_credits_at_this_bank  \
395                    none    rent                                       1   

                             job  \
395  skilled employee / official   

    number_of_people_bei

,feature,raw_value,woe_score,bin_labels,scores
0,status_of_existing_checking_account,0 <= ... < 200 DM,-28.044307,"[no checking account, ... >= 200 DM / salary a...","[79.98799835200919, 9.019990480758208, -28.044..."
12,age_in_years,25,-20.179989,"[[-inf, 25.50), [25.50, 29.00), [29.00, 34.45)...","[-20.17998946944557, -3.395069484957419, -2.68..."
2,credit_history,existing credits paid back duly till now,-4.352735,[critical account/ other credits existing (not...,"[36.731589901900854, -4.352734544235541, -6.91..."
8,personal_status_and_sex,male : single,-3.417786,"[male : divorced/separated, female : divorced/...","[14.128367945569327, 5.628274802268065, -3.417..."
19,foreign_worker,yes,-3.215227,"[no, yes]","[122.8590556257227, -3.215226730827078]"
16,job,skilled employee / official,-1.436829,"[unskilled - resident, skilled employee / offi...","[-3.787169505782917, -1.4368294661546412, 7.23..."
15,number_of_existing_credits_at_this_bank,1,0.025934,"[[-inf, 2.5), [2.5, +inf)]","[0.02593415807724011, -0.9534405918828143]"
17,number_of_people_being_liable_to_provide_maint...,1,0.033349,"[[-inf, 1.5), [1.5, +inf)]","[0.03334896772391867, -0.1757089187190512]"
9,other_debtors_or_guarantors,none,0.304793,"[guarantor, none, co-applicant]","[25.54576982951274, 0.30479335984908923, -29.4..."
10,present_residence_since,1,1.236701,"[[-inf, 1.5), [1.5, +inf)]","[1.236700823306226, -0.1901804579872671]"


    status_of_existing_checking_account duration_in_month  \
435                   0 <= ... < 200 DM                12   

                               credit_history           purpose credit_amount  \
435  existing credits paid back duly till now  radio/television          1484   

       savings_account_and_bonds present_employment_since  \
435  unknown/ no savings account       1 <= ... < 4 years   

    installment_rate_in_percentage_of_disposable_income  \
435                                                  2    

    personal_status_and_sex other_debtors_or_guarantors  \
435           male : single                        none   

    present_residence_since     property age_in_years other_installment_plans  \
435                       1  real estate           25                    none   

    housing number_of_existing_credits_at_this_bank  \
435     own                                       1   

                             job  \
435  skilled employee / official   

    nu

,feature,raw_value,woe_score,bin_labels,scores
4,credit_amount,12389,-84.642090,"[[-inf, 430.5), [430.5, 9569.0), [9569.0, 1084...","[1511.8272707540511, 2.7384035015945156, -52.8..."
1,duration_in_month,36,-37.145445,"[[-inf, 6.5), [6.5, 15.0), [15.0, 30.0), [30.0...","[91.89860265270823, 24.098673189918422, -7.089..."
0,status_of_existing_checking_account,0 <= ... < 200 DM,-28.044307,"[no checking account, ... >= 200 DM / salary a...","[79.98799835200919, 9.019990480758208, -28.044..."
14,housing,for free,-22.706023,"[own, rent, for free]","[9.7963473011284, -21.416366040408654, -22.706..."
11,property,unknown / no property,-17.465928,"[real estate, building society savings agreeme...","[12.020385793552764, 1.3122548234640619, -0.88..."
3,purpose,car (new),-14.754090,"[car (used), radio/television, domestic applia...","[42.162294807069216, 32.68489356619543, -0.0, ..."
2,credit_history,existing credits paid back duly till now,-4.352735,[critical account/ other credits existing (not...,"[36.731589901900854, -4.352734544235541, -6.91..."
8,personal_status_and_sex,male : single,-3.417786,"[male : divorced/separated, female : divorced/...","[14.128367945569327, 5.628274802268065, -3.417..."
19,foreign_worker,yes,-3.215227,"[no, yes]","[122.8590556257227, -3.215226730827078]"
16,job,skilled employee / official,-1.436829,"[unskilled - resident, skilled employee / offi...","[-3.787169505782917, -1.4368294661546412, 7.23..."


    status_of_existing_checking_account duration_in_month  \
563                   0 <= ... < 200 DM                36   

                               credit_history    purpose credit_amount  \
563  existing credits paid back duly till now  car (new)         12389   

       savings_account_and_bonds present_employment_since  \
563  unknown/ no savings account       1 <= ... < 4 years   

    installment_rate_in_percentage_of_disposable_income  \
563                                                  1    

    personal_status_and_sex other_debtors_or_guarantors  \
563           male : single                        none   

    present_residence_since               property age_in_years  \
563                       4  unknown / no property           37   

    other_installment_plans   housing number_of_existing_credits_at_this_bank  \
563                    none  for free                                       1   

                             job  \
563  skilled employee / official 